run high recall model on high trash count photos

In [ ]:
from mtrain.example_dir.core import load_npz
from mtrain.example_dir.defaults.negmask import default_negmask_learners
from mtrain.example_dir.bulk import to_example_dirs
import sys
import torch
from tqdm import tqdm
from mtrain.example_dir.defaults.smallnet import default_smallnet_learners
from mtrain.example_dir.iterdir import get_dirs
from mtrain.neg_mask.openai_clip import get_images_from_clip_file
from mtrain.example_dir import run_bulk_inference, create_dirs_for_images
from pathlib import Path
from mtrain.utils import *

CHUNKS_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi/chunks")
MODELS_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models")

def trash_mask(d, label):
    t = load_npz(d / f"negmask-trash-{label}.npz")
    o = load_npz(d / f"negmask-other-{label}.npz")
    return t > o

def get_high_trash_pixel_counts(dirs):
    counts = []
    for d in tqdm(dirs, desc="High trash pixel count"):
        mask = DiskBooleanMask.load(d / "mask-md.png")
        tmask = trash_mask(d, "md")
        final = mask & tmask
        counts.append((final.sum(), d))
    counts = list(reversed(sorted(counts)))
    
    return counts

def get_sorted_dirs():
    counts = []
    for chunk_name in range(0, 66):
        chunk_name = str(chunk_name)
        print("start chunk:", chunk_name)
        chunk_dir = CHUNKS_DIR / chunk_name
        dirs = list(get_dirs(chunk_dir))
        counts.extend(get_high_trash_pixel_counts(dirs))
        print("counts length", len(counts))
    return list(reversed(sorted(counts)))


def main():
    import sys

    smallnet = default_smallnet_learners(MODELS_DIR, ["md"], 8)
    negmask = default_negmask_learners(MODELS_DIR, ["other-high-recall"], 8)
    counts_and_dirs = get_sorted_dirs()
    dirs = [d[1] for d in counts_and_dirs if d[0] > 0]

    edirs = to_example_dirs(dirs, smallnet, negmask, True)
    for edir in tqdm(edirs, desc="Stage: other-high-recall"):
        o, t = edir.negmask_paths("other-high-recall", "md")

    
    for chunk_name in range(min_val, max_val):
        chunk_name = str(chunk_name)
        print("start chunk:", chunk_name)
        chunk_dir = CHUNKS_DIR / chunk_name
        dirs = list(get_dirs(chunk_dir))
        smallnet = default_smallnet_learners(MODELS_DIR, ["md"], 8)
        negmask = default_negmask_learners(MODELS_DIR, ["other-high-recall"], 8)
        edirs = to_example_dirs(dirs, smallnet, negmask, True)


# if __name__ == "__main__":
#     main()




In [ ]:
main()